In [1]:
get_ipython().system('pip install keepa')
import keepa
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from google.colab import userdata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.0 MB/s eta 0:00:00


In [2]:
accesskey = userdata.get('KEEPA_API_KEY')
api = keepa.Keepa(accesskey)

INFO:keepa.keepa_sync:Using key ending in 6ea4n5


In [3]:
# products = api.query(["B0F3PT1VBL", "B0D54JZTHY", "B0B1JPPG2L", "B00FLYWNYQ", "B00365DABC", "B0B5F9SZW7", "B09KR8P3L5", "B094DK6JNF", "B0BZXMXTT2", "B00W5D1MDE"])
products = api.query(['B0F3PT1VBL'], history=True)
product = products[0]

DEBUG:keepa.keepa_sync:Executing single product query
INFO:keepa.keepa_sync:1200 tokens remain
DEBUG:keepa.keepa_sync:Estimated time to complete 1 request(s) is 0.50 minutes
DEBUG:keepa.keepa_sync:	with a refill rate of 20 token(s) per minute
100%|██████████| 1/1 [00:00<00:00,  1.76it/s]


In [ ]:
# -----------------------------
# 1) Get 500 Electronics ASINs
# -----------------------------

ELECTRONICS_CATEGORY_ID = 172282

# Use the Electronics best sellers list directly
electronics_asins = api.best_sellers_query(ELECTRONICS_CATEGORY_ID)

# dedupe and keep first 500
electronics_asins = list(dict.fromkeys(electronics_asins))[:500]

print(f"Collected {len(electronics_asins)} Electronics ASINs")
print(electronics_asins[:10])

INFO:keepa.keepa_sync:930 tokens remain


Collected 500 Electronics ASINs
['B0D9ML2N4G', 'B0DGHMNQ5Z', 'B0GJTFXNRX', 'B0FQFB8FMG', 'B0D54JZTHY', 'B0DZ7871B8', 'B08R6S1M1K', 'B0DZ254SSR', 'B0DXXYS4BJ', 'B0D6GFNFJY']


In [ ]:
# -----------------------------
# 2) Helpers
# -----------------------------

KEEPA_EPOCH = datetime.datetime(2011, 1, 1) if "datetime" in globals() and hasattr(datetime, "datetime") else datetime(2011, 1, 1)

def keepa_to_datetime(keepa_minutes):
    return KEEPA_EPOCH + timedelta(minutes=int(keepa_minutes))

CSV_IDX = {
    "amazon": 0,
    "new": 1,
    "sales": 3,
    "count_new": 11,
}

PRICE_SERIES = {"amazon", "new"}

def get_csv_series(product, idx):
    csv_data = product.get("csv", None)
    if csv_data is None or idx >= len(csv_data):
        return None
    return csv_data[idx]

def parse_pair_series(series, name, convert_price=False):
    if not series:
        return pd.DataFrame(columns=["keepa_minutes", "datetime", name])

    rows = []
    for i in range(0, len(series), 2):
        if i + 1 >= len(series):
            break
        t = series[i]
        v = series[i + 1]

        if v == -1:
            parsed = None
        else:
            parsed = v / 100.0 if convert_price else v

        rows.append({
            "keepa_minutes": t,
            "datetime": keepa_to_datetime(t),
            name: parsed
        })

    return pd.DataFrame(rows)

def build_aligned_keepa_df(product):
    raw_cols = ["amazon", "new", "sales", "count_new"]
    dfs = []

    product_type = product.get("productType", 0)
    if product_type in {3, 4, 5}:  # inaccessible, invalid, variation parent
        return pd.DataFrame(columns=[
            "asin", "brand", "keepa_minutes", "datetime",
            "amazon", "new", "count_new", "sales_score"
        ])

    for name in raw_cols:
        idx = CSV_IDX[name]
        series = get_csv_series(product, idx)

        df = parse_pair_series(
            series,
            name=name,
            convert_price=(name in PRICE_SERIES)
        )
        dfs.append(df)

    timeline_sources = [df[["keepa_minutes", "datetime"]] for df in dfs if not df.empty]
    if not timeline_sources:
        return pd.DataFrame(columns=[
            "asin", "brand", "keepa_minutes", "datetime",
            "amazon", "new", "count_new", "sales_score"
        ])

    timeline = (
        pd.concat(timeline_sources, ignore_index=True)
        .drop_duplicates()
        .sort_values("keepa_minutes")
        .reset_index(drop=True)
    )

    out = timeline.copy()

    for df in dfs:
        if not df.empty:
            out = out.merge(df, on=["keepa_minutes", "datetime"], how="left")

    for col in raw_cols:
        if col not in out.columns:
            out[col] = pd.NA

    out[raw_cols] = out[raw_cols].ffill()

    # fallback: if NEW is missing, use AMAZON
    out["new"] = out["new"].fillna(out["amazon"])

    # convert sales rank to a model-friendlier score
    out["sales_score"] = -np.log1p(out["sales"])

    # drop raw sales
    out = out.drop(columns=["sales"])

    out.insert(0, "asin", product.get("asin"))
    out.insert(1, "brand", product.get("brand"))

    out = out[
        ["asin", "brand", "keepa_minutes", "datetime", "amazon", "new", "count_new", "sales_score"]
    ]

    return out

In [ ]:
# -----------------------------
# 3) Query the 500 ASINs in batches
# -----------------------------

all_dfs = []
batch_size = 25

for start in range(0, len(electronics_asins), batch_size):
    batch_asins = electronics_asins[start:start + batch_size]
    print(f"Querying batch {start // batch_size + 1} / {(len(electronics_asins) + batch_size - 1) // batch_size}")

    try:
        products = api.query(batch_asins, history=True)

        for product in products:
            try:
                df_product = build_aligned_keepa_df(product)
                if not df_product.empty:
                    all_dfs.append(df_product)
            except Exception as inner_e:
                print(f"Skipped ASIN {product.get('asin')} due to parsing error: {inner_e}")

    except Exception as batch_e:
        print(f"Batch failed for ASINs {batch_asins[:3]}... : {batch_e}")


DEBUG:keepa.keepa_sync:Executing 25 item product query
DEBUG:keepa.keepa_sync:Estimated time to complete 25 request(s) is 0.50 minutes
DEBUG:keepa.keepa_sync:	with a refill rate of 20 token(s) per minute


Querying batch 1 / 20


100%|██████████| 25/25 [00:02<00:00, 10.23it/s]
/tmp/ipykernel_263/973965959.py:95: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out[raw_cols] = out[raw_cols].ffill()
/tmp/ipykernel_263/973965959.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["new"] = out["new"].fillna(out["amazon"])
DEBUG:keepa.keepa_sync:Executing 25 item product query
DEBUG:keepa.keepa_sync:Estimated time to complete 25 request(s) is 0.50 minutes
DEBUG:keepa.keepa_sync:	with a refill rate of 20 token(s) per minute


Querying batch 2 / 20


100%|██████████| 25/25 [00:03<00:00,  6.54it/s]
/tmp/ipykernel_263/973965959.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["new"] = out["new"].fillna(out["amazon"])
/tmp/ipykernel_263/973965959.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["new"] = out["new"].fillna(out["amazon"])
/tmp/ipykernel_263/973965959.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_op

...

100%|██████████| 25/25 [00:03<00:00,  7.43it/s]
/tmp/ipykernel_263/973965959.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["new"] = out["new"].fillna(out["amazon"])
/tmp/ipykernel_263/973965959.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["new"] = out["new"].fillna(out["amazon"])
/tmp/ipykernel_263/973965959.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_op

Querying batch 19 / 20


100%|██████████| 25/25 [00:03<00:00,  7.36it/s]
/tmp/ipykernel_263/973965959.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["new"] = out["new"].fillna(out["amazon"])
/tmp/ipykernel_263/973965959.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["new"] = out["new"].fillna(out["amazon"])
/tmp/ipykernel_263/973965959.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_op

Querying batch 20 / 20


100%|██████████| 25/25 [00:03<00:00,  6.90it/s]
/tmp/ipykernel_263/973965959.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["new"] = out["new"].fillna(out["amazon"])
/tmp/ipykernel_263/973965959.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["new"] = out["new"].fillna(out["amazon"])
/tmp/ipykernel_263/973965959.py:98: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_op

In [ ]:
# -----------------------------
# 4) Combine and export to CSV
# -----------------------------

if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_df = final_df.sort_values(["asin", "datetime"]).reset_index(drop=True)

    output_path = "electronics_500_keepa_history.csv"
    final_df.to_csv(output_path, index=False)

    print("Final shape:", final_df.shape)
    print("Unique ASINs:", final_df["asin"].nunique())
    display(final_df.tail(20))
    print(f"Saved to {output_path}")
else:
    print("No data collected.")

/tmp/ipykernel_263/451822655.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(all_dfs, ignore_index=True)


Final shape: (2773072, 8)
Unique ASINs: 500


,asin,brand,keepa_minutes,datetime,amazon,new,count_new,sales_score
2773052,B0GQZR9SQ9,Apple,8006416,2026-03-23 00:16:00,628.0,628.0,NaN,-5.446737
2773053,B0GQZR9SQ9,Apple,8006578,2026-03-23 02:58:00,628.0,628.0,NaN,-5.451038
2773054,B0GQZR9SQ9,Apple,8006644,2026-03-23 04:04:00,628.0,628.0,NaN,-5.451038
2773055,B0GQZR9SQ9,Apple,8006706,2026-03-23 05:06:00,628.0,628.0,NaN,-5.451038
2773056,B0GQZR9SQ9,Apple,8007432,2026-03-23 17:12:00,628.0,628.0,NaN,-5.451038
2773057,B0GQZR9SQ9,Apple,8007802,2026-03-23 23:22:00,628.0,628.0,NaN,-5.505332
2773058,B0GQZR9SQ9,Apple,8007930,2026-03-24 01:30:00,628.0,628.0,NaN,-5.505332
2773059,B0GQZR9SQ9,Apple,8008612,2026-03-24 12:52:00,628.0,628.0,NaN,-5.590987
2773060,B0GQZR9SQ9,Apple,8008740,2026-03-24 15:00:00,628.0,628.0,NaN,-5.579730
2773061,B0GQZR9SQ9,Apple,8008964,2026-03-24 18:44:00,628.0,628.0,NaN,-5.572154


Saved to electronics_500_keepa_history.csv


In [4]:
# All keys associated with product['DATA']:
"""
AMAZON - Amazon price history
NEW - Marketplace/3rd party New price history - Amazon is considered to be part of the marketplace as well, so if Amazon has the overall lowest new (!) price, the marketplace new price in the corresponding time interval will be identical to the Amazon price (except if there is only one marketplace offer). Shipping and Handling costs not included!
USED - Marketplace/3rd party Used price history
SALES - Sales Rank history. Not every product has a Sales Rank.
LISTPRICE - List Price history
COLLECTIBLE - Collectible Price history
REFURBISHED - Refurbished Price history
NEW_FBM_SHIPPING -3rd party (not including Amazon) New price history including shipping costs, only fulfilled by merchant (FBM).
LIGHTNING_DEAL - 3rd party (not including Amazon) New price history including shipping costs, only fulfilled by merchant (FBM).
WAREHOUSE - Amazon Warehouse Deals price history. Mostly of used condition, rarely new.
NEW_FBA - Price history of the lowest 3rd party (not including Amazon/Warehouse) New offer that is fulfilled by Amazon
COUNT_NEW - New offer count history
COUNT_USED - Used offer count history
COUNT_REFURBISHED - Refurbished offer count history
COUNT_COLLECTIBLE - Collectible offer count history
RATING - The product’s rating history. A rating is an integer from 0 to 50 (e.g. 45 = 4.5 stars)
COUNT_REVIEWS - The product’s review count history.
BUY_BOX_SHIPPING - The price history of the buy box. If no offer qualified for the buy box the price has the value -1. Including shipping costs.
USED_NEW_SHIPPING - “Used - Like New” price history including shipping costs.
USED_VERY_GOOD_SHIPPING - “Used - Very Good” price history including shipping costs.
USED_GOOD_SHIPPING - “Used - Good” price history including shipping costs.
USED_ACCEPTABLE_SHIPPING - “Used - Acceptable” price history including shipping costs.
COLLECTIBLE_NEW_SHIPPING - “Collectible - Like New” price history including shipping costs.
COLLECTIBLE_VERY_GOOD_SHIPPING - “Collectible - Very Good” price history including shipping costs.
COLLECTIBLE_GOOD_SHIPPING - “Collectible - Good” price history including shipping costs.
COLLECTIBLE_ACCEPTABLE_SHIPPING - “Collectible - Acceptable” price history including shipping costs.
REFURBISHED_SHIPPING - Refurbished price history including shipping costs.
TRADE_IN - The trade in price history. Amazon trade-in is not available for every locale.
"""



'\nAMAZON - Amazon price history\nNEW - Marketplace/3rd party New price history - Amazon is considered to be part of the marketplace as well, so if Amazon has the overall lowest new (!) price, the marketplace new price in the corresponding time interval will be identical to the Amazon price (except if there is only one marketplace offer). Shipping and Handling costs not included!\nUSED - Marketplace/3rd party Used price history\nSALES - Sales Rank history. Not every product has a Sales Rank.\nLISTPRICE - List Price history\nCOLLECTIBLE - Collectible Price history\nREFURBISHED - Refurbished Price history\nNEW_FBM_SHIPPING -3rd party (not including Amazon) New price history including shipping costs, only fulfilled by merchant (FBM).\nLIGHTNING_DEAL - 3rd party (not including Amazon) New price history including shipping costs, only fulfilled by merchant (FBM).\nWAREHOUSE - Amazon Warehouse Deals price history. Mostly of used condition, rarely new.\nNEW_FBA - Price history of the lowest 3r

Keepa Query Documentation
[Link](https://keepaapi.readthedocs.io/en/latest/product_query.html)

# Mock Keepa API Endpoints

## GET `/mock/keepa/product/{asin}`

Returns product metadata.

**Response:**

```json
{
  "asin": "B0F3PT1VBL",
  "title": "Sony WH-1000XM6 Noise Cancelling Headphones",
  "brand": "Sony",
  "category": "Electronics"
}
```

---

## GET `/mock/keepa/product/{asin}/history`

Returns historical pricing data.

**Response:**

```json
{
  "asin": "B0F3PT1VBL",
  "price_history": [
    {
      "timestamp": "2024-03-01",
      "amazon_price": 398.99,
      "sales_rank": 215,
      "rating": 4.6,
      "review_count": 3205
    },
    {
      "timestamp": "2024-03-02",
      "amazon_price": 399.99,
      "sales_rank": 198,
      "rating": 4.6,
      "review_count": 3207
    }
    ...
  ]
}
```

---

## GET `/mock/keepa/product/{asin}/forecast`

Returns price forecast and buy/wait recommendation.

**Response:**

```json
{
  "asin": "B0F3PT1VBL",
  "forecast": [
    {
      "date": "2026-03-14",
      "predicted_price": 379.99,
      "confidence_lower": 365,
      "confidence_upper": 395,
      "recommendation": "WAIT"
    }
  ]
}
```